# Does one model fit the whole map?

Notebook 04 fitted a single spatial lag model to all 49 Columbus neighbourhoods.
That imposes an assumption worth taking seriously: that the same $\beta$ governs
the relation between income, housing value and crime everywhere in the city.

Spatial econometrics calls the failure of that assumption **spatial
heterogeneity**, and it is a different problem from spatial dependence. Dependence
is about how observations relate to each other; heterogeneity is about whether the
parameters are constant across space. A model can suffer from either, both, or
neither.

This notebook takes the discrete approach: split the map into regimes, let each
regime have its own coefficients, and test whether the split was worth making.

> Data provenance and citation are as in notebook 04.

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
Pkg.instantiate()

using SpatialEconometrics
using LinearAlgebra, SparseArrays, Statistics, Printf, Random
import StatsAPI: coef, coefnames, stderror, loglikelihood, nobs

In [ ]:
function read_columbus(dir)
    lines = readlines(joinpath(dir, "columbus.csv"))
    hdr   = split(strip(lines[1]), ',')
    vals  = [parse.(Float64, split(strip(l), ',')) for l in lines[2:end]]
    cols  = Dict(h => [v[j] for v in vals] for (j, h) in enumerate(hdr))
    W, _  = read_gal(joinpath(dir, "columbus.gal"); ids = Int.(cols["POLYID"]))
    return cols, W
end

cols, W = read_columbus(joinpath(@__DIR__, "data"))
N  = length(cols["CRIME"])
y  = cols["CRIME"]
X  = hcat(cols["INC"], cols["HOVAL"])
vn = ["income", "hoval"]

m_pool = fit_spatial(SAR(), W, X, y; varnames = vn, method = ExactLU())
@printf("pooled SAR: rho = %+.4f, logL = %.3f\n", m_pool.rho, loglikelihood(m_pool))

## 1. A regime split you can defend in advance

`NSA` is a dummy in the Columbus file marking the north side of the city. It is an
*exogenous* regime: the split is decided before looking at the residuals, which is
what keeps the subsequent test honest. Splitting on something the data suggested
would invalidate the distribution of the test statistic.

`ExogenousRegimes` takes integer labels starting at 1.

In [ ]:
labels = Int.(cols["NSA"]) .+ 1          # NSA is 0/1, regimes must start at 1
@printf("regime sizes: %s\n", [count(==(k), labels) for k in 1:2])

m_reg = fit_spatial_regime(W, X, y, ExogenousRegimes(labels);
                           varnames = vn, method = ExactLU())
println(m_reg)

The two regimes look different in an interesting way. In the north, income carries
the relation and housing value is insignificant; in the south it is the other way
round. A referee would find that an inviting story.

## 2. Is the split real?

`regime_chow_test` is a likelihood ratio test of the pooled model against the
regime model. The regime model nests the pooled one, so twice the log-likelihood
difference is $\chi^2$ with degrees of freedom equal to the extra parameters.

In [ ]:
println(regime_chow_test(m_pool, m_reg))

**It is not.** The likelihood barely moves, and with three extra parameters the
test does not come close to rejecting. The visibly different coefficient patterns
in section 1 are what two noisy subsamples of 25 and 24 observations look like.

This is the lesson of the notebook, and it is worth sitting with. Nothing about
the regime table *looked* like noise. Had we reported it without the test — or
chosen the split after seeing which one separated the coefficients — we would have
published a spatial regime effect that the data does not support.

## 3. Letting the data choose the regions

If you do not have an `NSA` to hand, the regions have to come from somewhere.
SKATER builds a minimum spanning tree over the contiguity graph and cuts it, so
every regime is guaranteed to be a *connected* region rather than an arbitrary
grouping of similar observations.

In [ ]:
sk = skater_regimes(W, hcat(y, X), 3; min_size = 5, rng = MersenneTwister(11))
@printf("SKATER regime sizes: %s\n", [count(==(k), sk.labels) for k in 1:3])

m_sk = fit_spatial_regime(W, X, y, sk; varnames = vn, method = ExactLU())
@printf("logL pooled = %.3f   logL SKATER = %.3f\n",
        loglikelihood(m_pool), loglikelihood(m_sk))
println(regime_chow_test(m_pool, m_sk))

Treat this number with suspicion, and the notebook says so deliberately. SKATER
chose the partition *using the data*, including `y`. The Chow test it feeds then
asks whether a split selected to separate the data succeeds in separating the
data. The nominal p-value is not the real one, and no correction is applied here.

Data-driven regionalisation is a legitimate exploratory tool, and a poor
confirmatory one. If you want to test a regime structure, pre-register it the way
`NSA` was used in section 1, or hold out a sample.

## 4. Exercises

1. Re-run section 1 with the `EW` (east/west) dummy instead of `NSA`. Does either
   split do better, and how would you report having tried both?
2. `min_size` controls the smallest SKATER region. Try 2, 5 and 10. How stable are
   the regimes, and what does the instability tell you about the number of
   regions the data can actually support?
3. The regime model estimates a single $\rho$ across regimes. Is that reasonable?
   Write down what it assumes about the spatial process at a regime boundary.
4. With 49 observations and 3 extra parameters, what effect size would the Chow
   test have detected at 80% power? Use `power_analysis` and decide whether
   section 2 is evidence of no break or merely absence of evidence.
5. Compare `spatial_aic(m_pool)` with `spatial_aic(m_reg)`. AIC and the LR test
   can disagree, since they trade off fit and parameters differently. Which would
   you report, and why?